# 05 LushProtein Retention Playbook — Two Bets  ·  *Finals cohort*

**A client recommendation: not a menu of tactics, but the two strategies we would stake the budget on.**

> **Data note:** built on the clean finals analysis pool (`EDA/outputs_finals/`): DQ-filtered, finals-eligible customers (2022+, excl. July/Nov acquisitions, elite, and 51%+ first-order discounts). The breadth and subscription tables are recomputed live from the finals customer base, so the numbers are the *defensible* post-presentation figures.

Notebooks `01`–`04` re-pointed the market basket analysis at one question: *which product combinations and sequences turn a one-and-done buyer into a repeat, subscribed, high-LTV customer?* This notebook **picks two bets**, because the finals data contains exactly **two dominant retention effects**, each far larger than any merchandising tweak. Every other idea (replenishment flows, sampler funnels, win-backs, margin guardrails) is a *mechanic that serves one of these two bets*, not a strategy. The bets:

1. **The Stack Ladder** — move single-product buyers up the *category-breadth* curve.
2. **The Subscription Flywheel** — convert the right routines into *subscriptions*.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EDA" / "outputs").exists() and (candidate / "product_mba").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs and product_mba")

PROJECT_ROOT = find_project_root()
EDA = PROJECT_ROOT / "EDA" / "outputs_finals"   # FINALS cohort
MBA = PROJECT_ROOT / "product_mba" / "outputs"
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

customers = pd.read_parquet(EDA / "customers.parquet")
customers["customer_id"] = customers["customer_id"].astype(str)
lines = pd.read_parquet(EDA / "lines_sku_analysis.parquet")
lines["customer_id"] = lines["customer_id"].astype(str)

anchors = pd.read_csv(MBA / "retention_anchor_products.csv")
scored_handle = pd.read_csv(MBA / "mba_rules_retention_scored_handle.csv")
seq = pd.read_csv(MBA / "mba_sequential_rules.csv")
gateway_flavor = pd.read_csv(MBA / "gateway_product_scorecard.csv")
triggers = pd.read_csv(MBA / "replenishment_triggers.csv")

BASE_REPEAT = customers["is_repeat"].mean()
BASE_SUB = customers["ever_subscribed"].mean()
print(f"Finals base: {len(customers):,} customers | repeat {BASE_REPEAT:.0%} | subscription {BASE_SUB:.0%}")

Finals base: 6,353 customers | repeat 23% | subscription 11%


## Why only two? The finals data has two dominant effects

In [2]:
# Breadth cliff (recomputed from finals lines + customers)
ncat = lines.dropna(subset=["product_category"]).groupby("customer_id")["product_category"].nunique()
cb = customers[["customer_id", "is_repeat", "total_revenue", "total_orders"]].copy()
cb["n_cat"] = cb["customer_id"].map(ncat).fillna(0).astype(int)
cb = cb[cb["n_cat"] >= 1]
cb["band"] = np.where(cb["n_cat"] >= 4, "4+ products",
                      cb["n_cat"].astype(str) + " product" + np.where(cb["n_cat"] == 1, "", "s"))
order = ["1 product", "2 products", "3 products", "4+ products"]
breadth = (cb.groupby("band").agg(customers=("customer_id", "size"), repeat_rate=("is_repeat", "mean"),
           avg_ltv=("total_revenue", "mean"), avg_orders=("total_orders", "mean"))
           .reindex(order).dropna(how="all"))

# Subscription split (recomputed from finals customers)
cs = customers.copy()
cs["seg"] = np.where(cs["ever_subscribed"], "Subscriber", "Non-subscriber")
sub_ltv = cs.groupby("seg").agg(customers=("customer_id", "size"), repeat_rate=("is_repeat", "mean"),
          avg_ltv=("total_revenue", "mean"), avg_orders=("total_orders", "mean"),
          avg_lifespan_days=("lifespan_days", "mean"))

print("EFFECT 1 — THE BREADTH CLIFF (distinct categories bought)")
for b in order:
    if b in breadth.index:
        r = breadth.loc[b]
        print(f"   {b:<11} {int(r.customers):>5,} cust | repeat {r.repeat_rate:5.0%} | "
              f"LTV ${r.avg_ltv:6.0f} | {r.avg_orders:.1f} orders")
o1, o3 = breadth.loc["1 product"], breadth.loc["3 products"]
print(f"   --> repeat {o1.repeat_rate:.0%} -> {o3.repeat_rate:.0%}; "
      f"LTV ${o1.avg_ltv:.0f} -> ${o3.avg_ltv:.0f} ({o3.avg_ltv/o1.avg_ltv:.1f}x) by adding categories")

sub, non = sub_ltv.loc["Subscriber"], sub_ltv.loc["Non-subscriber"]
print("\nEFFECT 2 — THE SUBSCRIPTION JACKPOT")
for nm, r in [("Subscriber", sub), ("Non-subscriber", non)]:
    print(f"   {nm:<14} {int(r.customers):>5,} | repeat {r.repeat_rate:5.0%} | LTV ${r.avg_ltv:6.0f} | "
          f"{r.avg_orders:.1f} orders | lifespan {r.avg_lifespan_days:.0f}d")
print(f"   --> a subscriber is worth {sub.avg_ltv/non.avg_ltv:.1f}x and lives "
      f"{sub.avg_lifespan_days/non.avg_lifespan_days:.1f}x longer")

EFFECT 1 — THE BREADTH CLIFF (distinct categories bought)
   1 product   3,681 cust | repeat   16% | LTV $   105 | 1.3 orders
   2 products  1,411 cust | repeat   33% | LTV $   146 | 1.8 orders
   3 products    445 cust | repeat   59% | LTV $   248 | 2.6 orders
   4+ products   157 cust | repeat   84% | LTV $   743 | 8.9 orders
   --> repeat 16% -> 59%; LTV $105 -> $248 (2.4x) by adding categories

EFFECT 2 — THE SUBSCRIPTION JACKPOT
   Subscriber       714 | repeat   67% | LTV $   251 | 3.4 orders | lifespan 196d
   Non-subscriber 5,639 | repeat   17% | LTV $   114 | 1.3 orders | lifespan 39d
   --> a subscriber is worth 2.2x and lives 5.1x longer


In [3]:
n_total = int(breadth["customers"].sum())
n_1cat = int(breadth.loc["1 product", "customers"])
n_nonsub = int(non.customers)
print(f"{n_1cat:,} of {n_total:,} customers ({n_1cat/n_total:.0%}) buy only ONE category")
print(f"{n_nonsub:,} of {len(customers):,} customers ({n_nonsub/len(customers):.0%}) have NEVER subscribed")
print("\n--> The prize is moving the customers we already have up these two curves.")

3,681 of 5,694 customers (65%) buy only ONE category
5,639 of 6,353 customers (89%) have NEVER subscribed

--> The prize is moving the customers we already have up these two curves.


## The size of the prize (illustrative LTV-gap math, not forecasts)

In [4]:
two = breadth.loc["2 products"]
gap_1to2 = two.avg_ltv - o1.avg_ltv
gap_sub = sub.avg_ltv - non.avg_ltv

print(f"STACK LADDER — addressable: {n_1cat:,} one-category buyers (gap to 2-cat = ${gap_1to2:.0f})")
for rate in (0.05, 0.10, 0.20):
    m = n_1cat * rate
    print(f"   move {rate:>3.0%} up one rung -> {m:,.0f} cust x ${gap_1to2:.0f} = +${m*gap_1to2:,.0f} LTV")

print(f"\nSUBSCRIPTION FLYWHEEL — addressable: {n_nonsub:,} non-subscribers (gap = ${gap_sub:.0f} each)")
for rate in (0.03, 0.05, 0.10):
    c = n_nonsub * rate
    print(f"   convert {rate:>3.0%} -> {c:,.0f} subscribers x ${gap_sub:.0f} = +${c*gap_sub:,.0f} LTV")

STACK LADDER — addressable: 3,681 one-category buyers (gap to 2-cat = $41)
   move  5% up one rung -> 184 cust x $41 = +$7,473 LTV
   move 10% up one rung -> 368 cust x $41 = +$14,946 LTV
   move 20% up one rung -> 736 cust x $41 = +$29,892 LTV

SUBSCRIPTION FLYWHEEL — addressable: 5,639 non-subscribers (gap = $137 each)
   convert  3% -> 169 subscribers x $137 = +$23,217 LTV
   convert  5% -> 282 subscribers x $137 = +$38,695 LTV
   convert 10% -> 564 subscribers x $137 = +$77,390 LTV


---
## MEGA-STRATEGY 1 — The Stack Ladder

> **Bet:**systematically walk single-product buyers up the category-breadth curve (1 → 2 → 3), using MBA to choose the *right* next product at the *right* moment.

### Why it is the best breadth bet — by the finals numbers
- **Biggest addressable segment** — the majority of customers sit on rung 1.
- **Largest, monotonic effect** — repeat rate and LTV step up at every rung and never flatten (see table above).
- **MBA makes it causal** — the **retention-anchor** table picks the next product by downstream stickiness; the **sequential rules** show real graduation paths; the **replenishment calendar** says when to prompt.
- **Cheapest to ship** — reuses computed rules, maps onto Shopify modules + Klaviyo/Recharge flows.

### What rolls up under it
Complete-Your-Stack module · replenishment-timed cross-sell · Discovery-Sampler/gateway front door · win-back that *widens* the basket · margin tie-breaker.

In [5]:
print(f"WHAT TO PUSH — retention-anchor products (lift over the {BASE_REPEAT:.0%} baseline repeat rate):")
print(anchors[anchors["level"] == "handle"].head(6)[
    ["item", "total_co_orders", "wtd_repeat_uplift", "wtd_subscription_uplift", "wtd_ltv_ratio"]].to_string(index=False))

print("\nGRADUATION PATHS — what customers reach for NEXT (sequential, handle, lift>=1.2):")
g = seq[(seq["level"] == "handle") & (seq["antecedent"] != seq["consequent"]) &
        (seq["next_order_lift"] >= 1.2)].sort_values("next_order_lift", ascending=False)
print(g.head(8)[["antecedent", "consequent", "co_customers", "next_order_confidence", "next_order_lift"]].to_string(index=False))

print("\nWHEN TO PROMPT — replenishment calendar (fire ~7d before run-out):")
cal = triggers.dropna(subset=["handle"]).sort_values("trigger_day")
for _, r in cal.head(8).iterrows():
    x = r["recommended_cross_sell"]
    tail = f" + introduce {x}" if isinstance(x, str) and x == x else " (reorder reminder)"
    print(f"   Day {int(r['trigger_day']):>3}: reorder {r['handle']}{tail}")

WHAT TO PUSH — retention-anchor products (lift over the 23% baseline repeat rate):
                               item  total_co_orders  wtd_repeat_uplift  wtd_subscription_uplift  wtd_ltv_ratio
         green-tea-extract-capsules               58               0.37                     0.09          13.17
       pureburn-fat-burner-capsules               58               0.37                     0.09          13.17
lean-protein-peach-oolong-pre-order               30               0.34                     0.09           1.77
                       lean-protein              554               0.23                     0.15           2.26
           lushprotein-clear-shaker              878               0.23                     0.11           3.38
                  discovery-sampler               40               0.21                     0.04          11.64

GRADUATION PATHS — what customers reach for NEXT (sequential, handle, lift>=1.2):
                               antecedent         

---
## MEGA-STRATEGY 2 — The Subscription Flywheel

> **Bet:** convert the *routines that already behave like subscriptions* into subscribe-and-save, defaulting the anchor item to subscription at the second purchase.

### Why it is the best value bet — by the finals numbers
- **Highest value per conversion** — a subscriber's LTV, repeat rate, order count and lifespan all dwarf a non-subscriber's (see table above). Nothing else moves a single customer this much.
- **Large untapped headroom** — the overwhelming majority have never subscribed.
- **MBA targets *who* will convert** — the pairs below are several× more subscribed than the baseline *before* we ask, so subscribe-and-save can be aimed (protecting margin) rather than blanket-discounted.
- **It compounds with Strategy 1** — a subscription is a locked-in stack; the ladder is the on-ramp, the flywheel is the lock-in.

In [6]:
sub_combos = (scored_handle.sort_values("subscription_uplift", ascending=False)
              .drop_duplicates(subset=["antecedent", "consequent"]).head(8))
print(f"ROUTINES PRIMED TO SUBSCRIBE (baseline subscription = {BASE_SUB:.0%}):")
print(sub_combos[["antecedent", "consequent", "co_orders", "both_item_pct_subscribed",
                  "subscription_uplift", "ltv_ratio"]].to_string(index=False))

print("\nSAMPLER / TRIAL AS THE SUBSCRIPTION ON-RAMP (sequential graduation):")
samp = seq[(seq["antecedent"].str.contains("single|sampler|sachet", case=False, na=False)) &
           (seq["antecedent"] != seq["consequent"])].sort_values("next_order_lift", ascending=False)
print(samp.head(5)[["antecedent", "consequent", "co_customers", "next_order_confidence",
                    "next_order_lift"]].to_string(index=False) if len(samp) else "   (none cleared thresholds)")

ROUTINES PRIMED TO SUBSCRIBE (baseline subscription = 11%):
                               antecedent                                consequent  co_orders  both_item_pct_subscribed  subscription_uplift  ltv_ratio
                 lushprotein-clear-shaker                              lean-protein        277                      0.26                 0.15       2.26
                             lean-protein                  lushprotein-clear-shaker        277                      0.26                 0.15       2.26
      lean-protein-peach-oolong-pre-order                  lushprotein-clear-shaker         30                      0.20                 0.09       1.77
             pureburn-fat-burner-capsules                green-tea-extract-capsules         29                      0.20                 0.09      13.17
               green-tea-extract-capsules              pureburn-fat-burner-capsules         29                      0.20                 0.09      13.17
lushprotein-lean-prote

---
## Why these two — and not the other ideas

| Candidate | Verdict | Reason (finals data) |
|---|---|---|
| **Stack Ladder** | **Bet** | Largest segment (most customers on rung 1) × largest monotonic effect (repeat & LTV step up every rung). |
| **Subscription Flywheel** | **Bet** | Largest value per head (multi-× LTV & lifespan) × large untapped base. |
| Replenishment flows | *Mechanic of Bet 1/2* | The *timing* that fires the cross-sell/reorder. |
| Sampler funnel | *Mechanic of Bet 1/2* | The *front door* feeding both ladders (single-serve → full-size at ~2.7× lift). |
| Win-back | *Mechanic of Bet 1* | Re-entry that should widen the basket toward rung 2. |
| Margin guardrail | *Tie-breaker only* | Margin is directional (see `00`); a constraint, not a strategy. |

**They reinforce each other.** Breadth builds the multi-product routine; subscription locks it in. The end-state both bets aim at — a 4+ category subscriber — is already the highest-repeat, highest-LTV customer in the finals data. Run them as one funnel: **ladder up, then lock in.**

## Measurement & limitations
- **Measure with holdouts.** Primary KPIs: % reaching 2+/3+ categories (Bet 1) and subscription conversion (Bet 2) — not AOV. The tables above are the finals baselines to beat.
- **Limitations:** ~25–38% of finals orders are multi-item, so affinity is under-counted; margin is directional (`00`); the finals cohort is deliberately narrow (2022+, excl. Jul/Nov acquisitions & elite), so absolute rates are conservative vs the full base; sequential rules cover first→second order only; analysis pooled across SG/HK/MY. Prize figures are LTV-gap math, not forecasts.

**Bottom line:** on the clean finals cohort, the same MBA — re-pointed from "biggest basket" to "stickiest basket" — converges on two bets, **ladder up, then lock in**, addressing most of the base and worth six figures of LTV from single-digit conversion of customers LushProtein already owns.